In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
from pathlib import Path

import torch
import zarr
from zarr.codecs import BloscCodec
from PIL import Image
from tqdm.auto import tqdm
import pyvista as pv
import matplotlib
matplotlib.use("Agg") if os.environ.get("PYVISTA_OFF_SCREEN") else None
%matplotlib inline

pv.set_jupyter_backend("static" if os.environ.get("PYVISTA_OFF_SCREEN") else "trame")

from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.semantics.features import MaskCLIPExtractor, DINOFeatureExtractor, Talk2DinoExtractor
from collab_splats.semantics.compression import FeatureAutoencoder
from collab_splats.pointcloud.utils import lift_features
from collab_splats.utils.visualization import (
    pointcloud_to_polydata,
    visualize_splat,
    PCD_KWARGS,
    VIZ_KWARGS,
)

## §0 — Configuration

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
METHOD         = "vggtx"    # "vggtx" | "mapanything"
LATENT_DIM     = 13
QUERY_POSITIVE = ["tree"]
QUERY_NEGATIVE = ["ground"]
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# Prefer BA reconstruction if available; fall back to raw feedforward result
_ba   = CACHE_DIR / METHOD / "ba" / "reconstruction.zarr"
_raw  = CACHE_DIR / METHOD / "reconstruction.zarr"
RECON = _ba if _ba.exists() else _raw
_base = RECON.parent  # directory that holds lifted.zarr outputs for this reconstruction

# Scene-level semantics dir — AE weights are method-independent, reused across reconstructions
SEMANTICS_DIR    = CACHE_DIR / "semantics"
AE_MASKCLIP      = SEMANTICS_DIR / "maskclip"
AE_TALK2DINO     = SEMANTICS_DIR / "talk2dino"

# Extractor-named cache paths — never collide when switching extractors or methods
LIFTED_MASKCLIP  = _base / "lifted_maskclip.zarr"
LIFTED_TALK2DINO = _base / "lifted_talk2dino.zarr"

# Semantic visualization kwargs — extend PCD_KWARGS with scalar coloring
SEMANTIC_MESH_KWARGS = {**PCD_KWARGS, "scalars": "semantic", "cmap": "viridis", "rgb": False}

assert RECON.exists(), (
    f"Reconstruction not found at {RECON}. "
    "Run 02_pointcloud/feedforward_methods first."
)
print(f"Device: {DEVICE}  |  RECON: {RECON}")
print(f"BA: {'yes' if _ba.exists() else 'no — using raw feedforward result'}")
print(f"MaskCLIP cache:  {LIFTED_MASKCLIP}")
print(f"Talk2DINO cache: {LIFTED_TALK2DINO}")
print(f"AE MaskCLIP:     {AE_MASKCLIP}")
print(f"AE Talk2DINO:    {AE_TALK2DINO}")

## §1 — Load Reconstruction

Loads the cached feedforward reconstruction from zarr. Produces a `FeedforwardResult` with
world-space 3D points, per-point colors, pixel source indices, and source image paths.
Run `02_pointcloud/feedforward_methods` first if the assertion above failed.

In [ ]:
# Load reconstruction from zarr cache
out = FeedforwardResult.load_zarr(RECON)
print(f"Loaded reconstruction: {out.points.shape[0]:,} pts  |  {len(out.image_paths)} frames")

## §2 — Inspect Outputs

Confirms the three arrays needed for feature lifting: `pts3d` (3D positions), `pixel_indices`
(source frame/row/col per point), and `colors` (RGB). `pixel_indices` is the bridge between
the 2D feature maps and the 3D point cloud.

In [ ]:
pts3d         = out.points         # (P, 3) float32
pixel_indices = out.pixel_indices  # (P, 3) int32  [frame_id, row, col]
colors        = out.colors         # (P, 3) uint8

print(f"pts3d:         {pts3d.shape}")
print(f"pixel_indices: {pixel_indices.shape}")
print(f"colors:        {colors.shape}")
print(f"image_paths:   {len(out.image_paths)} frames")

## §3 — MaskCLIP: Extract, Compress & Lift

Extracts MaskCLIP patch features from every source frame, then fits a `FeatureAutoencoder`
to compress them from full-dim to `LATENT_DIM` before lifting onto the point cloud.

MaskCLIP patch features are purely appearance-based — they lack structural grounding.
A DINOv2 regularization branch during AE training improves the latent space geometry
by pulling structurally similar patches together. Talk2DINO (§6) already carries CLIP
grounding, so no regularization is needed there.

**Cache hit:** `feat_mc` is loaded directly — extraction and compression are skipped.

In [ ]:
# Load frames once — reused in §6
imgs = [Image.open(p).convert("RGB") for p in tqdm(out.image_paths, desc="Loading frames")]

# Init MaskCLIP — used for both extraction and scoring in §4
maskclip = MaskCLIPExtractor(device=DEVICE)

########################################################################

if LIFTED_MASKCLIP.exists() and AE_MASKCLIP.exists():
    ae_mc = FeatureAutoencoder.load(AE_MASKCLIP).to(DEVICE)
    compressed_mc = torch.from_numpy(np.asarray(zarr.open_group(store=str(LIFTED_MASKCLIP), mode="r")["features"][:]))
    feat_mc = ae_mc.per_point_decode(compressed_mc.to(DEVICE)).detach().cpu()
    print(f"Loaded from cache: {compressed_mc.shape} → decoded {feat_mc.shape}")
else:
    # DINOv2 only needed for AE regularization; init here to skip on cache hit
    dinov2 = DINOFeatureExtractor(device=DEVICE)

    mc_maps = maskclip.forward(imgs)  # list of (D, H_p, W_p)
    dv_maps = dinov2.forward(imgs)    # list of (384, H_p, W_p)

    # Train AE: MaskCLIP reconstruction + DINOv2 regularization branch
    D = mc_maps[0].shape[0]
    ae_mc = FeatureAutoencoder(D, LATENT_DIM, regularization_kwargs={"branches": {"dinov2": 384}, "weight": 0.1})
    ae_mc.fit(
        torch.cat([fm.flatten(1).T for fm in mc_maps]).to(DEVICE),
        reg_targets={"dinov2": torch.cat([fm.flatten(1).T for fm in dv_maps]).to(DEVICE)},
    )

    # Compress each frame map, lift to 3D
    compressed = [ae_mc.encode(fm.to(DEVICE)).detach().cpu() for fm in mc_maps]
    compressed_mc = lift_features(compressed, out)          # (P, LATENT_DIM)
    feat_mc = ae_mc.per_point_decode(compressed_mc.to(DEVICE)).detach().cpu()

    # Save compressed features and AE weights
    g = zarr.open_group(store=str(LIFTED_MASKCLIP), mode="w")
    g.create_array("features", data=compressed_mc.numpy(), chunks=compressed_mc.shape, compressors=BloscCodec(cname="lz4"))
    ae_mc.save(AE_MASKCLIP)
    print(f"Saved: compressed {compressed_mc.shape} → {LIFTED_MASKCLIP}")

print(f"feat_mc: {feat_mc.shape}")

## §4 — MaskCLIP: Text Queries → Per-Point Scores

Scores each 3D point against the configured text queries. `score_queries` returns a
contrastive softmax score in [0, 1] — higher means the point matches the positive queries.
`compute_similarity` returns raw cosine similarities per query for multi-query inspection.

In [ ]:
feat_mc_dev  = feat_mc.to(DEVICE)
scores_mc    = maskclip.score_queries(feat_mc_dev, positive=QUERY_POSITIVE, negative=QUERY_NEGATIVE, temperature=0.05).detach().cpu().numpy()
per_query_mc = maskclip.compute_similarity(feat_mc_dev, QUERY_POSITIVE).detach().T.cpu().numpy()

print(f"scores_mc:    {scores_mc.shape}  min={scores_mc.min():.3f}  max={scores_mc.max():.3f}")
print(f"per_query_mc: {per_query_mc.shape}")

## §5 — MaskCLIP: Interactive 3D Viewer

Attaches semantic scores and per-query arrays to the point cloud. Use the scalar selector
in the side panel to switch between the RGB view and each semantic query.

In [ ]:
cloud_mc = pointcloud_to_polydata(
    pts3d,
    RGB=colors,
    semantic=scores_mc,
    **{q.replace(" ", "_"): per_query_mc[:, i] for i, q in enumerate(QUERY_POSITIVE)},
)

pl = visualize_splat(cloud_mc, mesh_kwargs=SEMANTIC_MESH_KWARGS, viz_kwargs=VIZ_KWARGS)
pl.show()

## §6 — Talk2DINO: Extract, Compress & Lift

Runs the same extract → compress → lift pipeline with Talk2DINO patch features.
Talk2DINO is CLIP-grounded — its patch features already carry structural and semantic
information from CLIP pre-training. No DINOv2 regularization branch is needed;
the AE trains on Talk2DINO patches directly.

**Cache hit:** `feat_t2d` is loaded directly — extraction and compression are skipped.

In [ ]:
# Init Talk2DINO — used for both extraction and scoring in §7
talk2dino = Talk2DinoExtractor(device=DEVICE)

########################################################################

if LIFTED_TALK2DINO.exists() and AE_TALK2DINO.exists():
    ae_t2d = FeatureAutoencoder.load(AE_TALK2DINO).to(DEVICE)
    compressed_t2d = torch.from_numpy(np.asarray(zarr.open_group(store=str(LIFTED_TALK2DINO), mode="r")["features"][:]))
    feat_t2d = ae_t2d.per_point_decode(compressed_t2d.to(DEVICE)).detach().cpu()
    print(f"Loaded from cache: {compressed_t2d.shape} → decoded {feat_t2d.shape}")
else:
    t2d_maps = talk2dino.forward(imgs)  # list of (D_t, H_p, W_p)

    # Train plain AE — no regularization needed (CLIP-grounded)
    D_t = t2d_maps[0].shape[0]
    ae_t2d = FeatureAutoencoder(D_t, LATENT_DIM)
    ae_t2d.fit(torch.cat([fm.flatten(1).T for fm in t2d_maps]).to(DEVICE))

    # Compress each frame map, lift to 3D
    compressed = [ae_t2d.encode(fm.to(DEVICE)).detach().cpu() for fm in t2d_maps]
    compressed_t2d = lift_features(compressed, out)         # (P, LATENT_DIM)
    feat_t2d = ae_t2d.per_point_decode(compressed_t2d.to(DEVICE)).detach().cpu()

    # Save compressed features and AE weights
    g = zarr.open_group(store=str(LIFTED_TALK2DINO), mode="w")
    g.create_array("features", data=compressed_t2d.numpy(), chunks=compressed_t2d.shape, compressors=BloscCodec(cname="lz4"))
    ae_t2d.save(AE_TALK2DINO)
    print(f"Saved: compressed {compressed_t2d.shape} → {LIFTED_TALK2DINO}")

print(f"feat_t2d: {feat_t2d.shape}")

## §7 — Talk2DINO: Text Queries → Per-Point Scores

Same query scoring as §4 but using the Talk2DINO text encoder. Both extractors implement
`score_queries` and `compute_similarity` via `BaseQueryableExtractor`.

In [ ]:
feat_t2d_dev  = feat_t2d.to(DEVICE)
scores_t2d    = talk2dino.score_queries(feat_t2d_dev, positive=QUERY_POSITIVE, negative=QUERY_NEGATIVE, temperature=0.05).detach().cpu().numpy()
per_query_t2d = talk2dino.compute_similarity(feat_t2d_dev, QUERY_POSITIVE).detach().T.cpu().numpy()

print(f"scores_t2d:    {scores_t2d.shape}  min={scores_t2d.min():.3f}  max={scores_t2d.max():.3f}")
print(f"per_query_t2d: {per_query_t2d.shape}")

## §8 — Talk2DINO: Interactive 3D Viewer

Renders Talk2DINO semantic scores on the point cloud. Use the scalar selector to switch
between RGB and per-query views. Compare visually with the MaskCLIP result in §5.

In [ ]:
cloud_t2d = pointcloud_to_polydata(
    pts3d,
    RGB=colors,
    semantic=scores_t2d,
    **{q.replace(" ", "_"): per_query_t2d[:, i] for i, q in enumerate(QUERY_POSITIVE)},
)

pl = visualize_splat(cloud_t2d, mesh_kwargs=SEMANTIC_MESH_KWARGS, viz_kwargs=VIZ_KWARGS)
pl.show()